# Station 04｜強化學習：Connect X 智慧對手

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/day6_ai_solution_lab/04_rl_connectx_agent.ipynb)

**客戶任務：** 建立不下非法棋、可量測勝率與決策時間的 Connect X 對手。

- Kaggle 環境：[Connect X](https://www.kaggle.com/competitions/connect-x)
- 重要：Connect X 是互動環境與規則，不是一般靜態資料集。
- 輪轉主流程使用純 Python 輕量模擬器，避免安裝大型套件拖慢全班；最後另用 Kaggle environment 驗證一次。
- 本站比較 Random、Center-first、One-step Lookahead；DQN checkpoint 留給選案完整實作。


## 1. 純 Python Connect X 環境


In [ ]:
from dataclasses import dataclass
import random
import time

import numpy as np
import pandas as pd

SEED = 20260719
ROWS, COLUMNS, INAROW = 6, 7, 4
random.seed(SEED)
np.random.seed(SEED)

def legal_columns(board):
    return [column for column in range(COLUMNS) if board[0, column] == 0]

def drop_piece(board, column, player):
    if column not in legal_columns(board):
        return False
    row = max(row for row in range(ROWS) if board[row, column] == 0)
    board[row, column] = player
    return True

def has_won(board, player):
    for row in range(ROWS):
        for column in range(COLUMNS):
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                cells = [(row + step * dr, column + step * dc) for step in range(INAROW)]
                if all(0 <= r < ROWS and 0 <= c < COLUMNS for r, c in cells):
                    if all(board[r, c] == player for r, c in cells):
                        return True
    return False

def board_full(board):
    return not legal_columns(board)


## 2. 三個 Agent 與非法行動遮罩


In [ ]:
def random_agent(board, player, rng):
    return rng.choice(legal_columns(board))

def center_agent(board, player, rng):
    legal = legal_columns(board)
    return min(legal, key=lambda column: (abs(column - COLUMNS // 2), rng.random()))

BLOCK_OPPONENT = True  # TODO(學員必改)：先用 False，再改成 True 比較勝率

def lookahead_agent(board, player, rng):
    legal = legal_columns(board)  # 這就是非法行動 mask
    opponent = 2 if player == 1 else 1

    for column in legal:
        candidate = board.copy()
        drop_piece(candidate, column, player)
        if has_won(candidate, player):
            return column

    if BLOCK_OPPONENT:
        for column in legal:
            candidate = board.copy()
            drop_piece(candidate, column, opponent)
            if has_won(candidate, opponent):
                return column

    ordered = sorted(legal, key=lambda column: abs(column - COLUMNS // 2))
    return ordered[0]


## 3. 多局評估：輪替先後手


In [ ]:
def play_game(agent_a, agent_b, seed, a_starts=True):
    rng = random.Random(seed)
    board = np.zeros((ROWS, COLUMNS), dtype=np.int8)
    agents = {1: agent_a if a_starts else agent_b, 2: agent_b if a_starts else agent_a}
    a_player = 1 if a_starts else 2
    illegal_actions = 0
    decision_times = []

    for turn in range(ROWS * COLUMNS):
        player = 1 if turn % 2 == 0 else 2
        started = time.perf_counter()
        action = agents[player](board.copy(), player, rng)
        decision_times.append(time.perf_counter() - started)
        if not drop_piece(board, action, player):
            illegal_actions += 1
            winner = 2 if player == 1 else 1
            break
        if has_won(board, player):
            winner = player
            break
        if board_full(board):
            winner = 0
            break
    else:
        winner = 0

    outcome_a = "draw" if winner == 0 else ("win" if winner == a_player else "loss")
    return {
        "outcome_a": outcome_a,
        "illegal_actions": illegal_actions,
        "mean_decision_ms": 1000 * float(np.mean(decision_times)),
    }

def tournament(agent_a, agent_b, games=100):
    results = [
        play_game(agent_a, agent_b, SEED + game, a_starts=(game % 2 == 0))
        for game in range(games)
    ]
    frame = pd.DataFrame(results)
    counts = frame["outcome_a"].value_counts()
    return {
        "games": games,
        "wins": int(counts.get("win", 0)),
        "draws": int(counts.get("draw", 0)),
        "losses": int(counts.get("loss", 0)),
        "win_rate": float((frame["outcome_a"] == "win").mean()),
        "illegal_action_rate": float(frame["illegal_actions"].sum() / games),
        "mean_decision_ms": float(frame["mean_decision_ms"].mean()),
    }

comparison = pd.DataFrame({
    "Random vs Random": tournament(random_agent, random_agent),
    "Center vs Random": tournament(center_agent, random_agent),
    "Lookahead vs Random": tournament(lookahead_agent, random_agent),
    "Lookahead vs Center": tournament(lookahead_agent, center_agent),
}).T
display(comparison.round(3))


## 4. 必測棋盤：能贏、要擋、欄位已滿


In [ ]:
def action_for(board_rows, player=1):
    board = np.array(board_rows, dtype=np.int8)
    action = lookahead_agent(board, player, random.Random(SEED))
    return action, legal_columns(board)

win_board = np.zeros((ROWS, COLUMNS), dtype=np.int8)
win_board[-1, 0:3] = 1
block_board = np.zeros((ROWS, COLUMNS), dtype=np.int8)
block_board[-1, 0:3] = 2
full_center = np.zeros((ROWS, COLUMNS), dtype=np.int8)
full_center[:, COLUMNS // 2] = [1, 2, 1, 2, 1, 2]

tests = {
    "win_next": action_for(win_board),
    "block_next": action_for(block_board),
    "center_full": action_for(full_center),
}
print(tests)
assert tests["win_next"][0] == 3
if BLOCK_OPPONENT:
    assert tests["block_next"][0] == 3
assert COLUMNS // 2 not in tests["center_full"][1]
print("Boundary tests: PASS")


## 5. Kaggle environment 相容驗證（選用）

這格會安裝較大的 `kaggle-environments`，可留到輪轉最後或由講師示範。


In [ ]:
RUN_KAGGLE_ENV_CHECK = False
if RUN_KAGGLE_ENV_CHECK:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle-environments==1.32.0"])
    from kaggle_environments import make

    def kaggle_lookahead(observation, configuration):
        board = np.array(observation.board, dtype=np.int8).reshape(configuration.rows, configuration.columns)
        return int(lookahead_agent(board, observation.mark, random.Random(SEED)))

    env = make("connectx", debug=True)
    env.run([kaggle_lookahead, "random"])
    print("Kaggle environment status:", env.state[-1][0].status, env.state[-1][1].status)
else:
    print("Skipped in rotation mode. Set RUN_KAGGLE_ENV_CHECK=True for official environment validation.")


## 6. 課堂暫時 Demo：決策 API（選用）


In [ ]:
%pip -q install gradio==6.20.0
import gradio as gr

MODEL_VERSION = "station04-lookahead-v1"

def choose_column(board_json, player):
    try:
        values = np.array(board_json, dtype=np.int8).reshape(ROWS, COLUMNS)
    except Exception:
        return {"error": "請輸入 6x7 的 0/1/2 二維陣列", "model_version": MODEL_VERSION}
    legal = legal_columns(values)
    if not legal:
        return {"error": "棋盤已滿", "model_version": MODEL_VERSION}
    started = time.perf_counter()
    action = lookahead_agent(values, int(player), random.Random(SEED))
    elapsed_ms = 1000 * (time.perf_counter() - started)
    return {
        "column_zero_based": int(action),
        "legal": action in legal,
        "decision_ms": round(elapsed_ms, 3),
        "model_version": MODEL_VERSION,
    }

demo = gr.Interface(
    fn=choose_column,
    inputs=[gr.JSON(value=np.zeros((ROWS, COLUMNS), dtype=int).tolist(), label="棋盤"), gr.Radio([1, 2], value=1, label="Agent mark")],
    outputs=gr.JSON(label="決策"),
    title="Connect X 決策 API（課堂暫時 Demo）",
)
print("需要介面時再執行：demo.launch(share=True)")


## 7. 下載迷你實驗卡


In [ ]:
experiment_card = f'''# RL 站迷你實驗卡

- 組別：請填寫
- 我修改了：BLOCK_OPPONENT = {BLOCK_OPPONENT}
- 原本結果：請貼上修改前勝率／非法行動率／決策時間
- 修改後結果：請貼上修改後勝率／非法行動率／決策時間
- 最大失敗棋盤：請填寫
- Seed：{SEED}
- 模型版本：{MODEL_VERSION}
- 限制：本站是規則式 policy 體驗；DQN 需在完整實作載入有來源的 checkpoint。
'''
output_path = Path("/content/station04_rl_experiment_card.md")
output_path.write_text(experiment_card, encoding="utf-8")
print(experiment_card)
print("Saved:", output_path)
